In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64
import pandas as pd

# Configure the CRUD module import
from animal_shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# Update these if your MongoDB user credentials are different
username = "aacuser"
password = "REDACTED_SET_LOCALLY"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

def query_to_dataframe(query):
    """Run a MongoDB query through the CRUD module and return a clean DataFrame."""
    df = pd.DataFrame.from_records(db.read(query))
    if '_id' in df.columns:
        df.drop(columns=['_id'], inplace=True)
    return df

# Start with the full, unfiltered data set
df = query_to_dataframe({})

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Grazioso Salvare logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Center(html.B(html.H1('CS-340 Dashboard'))),

    html.Center([
        html.A(
            html.Img(
                src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style={'height': '250px'}
            ),
            href='https://www.snhu.edu',
            target='_blank'
        ),
        html.H3('Created by Savanna Neace')
    ]),

    html.Hr(),

    html.Div([
        html.H3('Rescue Type Filter'),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value='reset',
            inline=True,
            inputStyle={'margin-left': '15px', 'margin-right': '5px'}
        )
    ]),

    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0],
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'minWidth': '100px',
            'width': '150px',
            'maxWidth': '200px',
            'whiteSpace': 'normal'
        },
        style_header={
            'fontWeight': 'bold'
        }
    ),

    html.Br(),
    html.Hr(),

    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
                style={'width': '50%'}
            ),
            html.Div(
                id='map-id',
                className='col s12 m6',
                style={'width': '50%'}
            )
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    """Filter the dashboard data according to the selected rescue type."""

    if filter_type == 'water':
        query = {
            'animal_type': 'Dog',
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156},
            'breed': {'$regex': 'Labrador Retriever|Chesapeake Bay Retriever|Newfoundland', '$options': 'i'}
        }

    elif filter_type == 'mountain':
        query = {
            'animal_type': 'Dog',
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156},
            'breed': {'$regex': 'German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler', '$options': 'i'}
        }

    elif filter_type == 'disaster':
        query = {
            'animal_type': 'Dog',
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300},
            'breed': {'$regex': 'Doberman Pinscher|German Shepherd|Golden Retriever|Bloodhound|Rottweiler', '$options': 'i'}
        }

    else:
        query = {}

    filtered_df = query_to_dataframe(query)
    return filtered_df.to_dict('records')


@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):
    """Display a pie chart of breeds from the currently filtered data."""

    if viewData is None or len(viewData) == 0:
        return [html.H4('No data available for this filter.')]

    dff = pd.DataFrame.from_dict(viewData)

    if 'breed' not in dff.columns or dff.empty:
        return [html.H4('No breed data available.')]

    return [
        dcc.Graph(
            figure=px.pie(
                dff,
                names='breed',
                title='Preferred Rescue Dog Breeds'
            )
        )
    ]


@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        selected_columns = []

    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
def update_map(viewData, index):
    """Display the selected animal on the geolocation map."""

    if viewData is None or len(viewData) == 0:
        return [html.H4('No map data available for this filter.')]

    dff = pd.DataFrame.from_dict(viewData)

    required_columns = {'location_lat', 'location_long', 'breed', 'name'}
    if not required_columns.issubset(dff.columns):
        return [html.H4('Map cannot be displayed because location columns are missing.')]

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    if row >= len(dff):
        row = 0

    animal_name = dff.iloc[row]['name']
    if pd.isna(animal_name) or animal_name == '':
        animal_name = 'No name listed'

    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id='base-layer-id'),
                dl.Marker(
                    position=[dff.iloc[row]['location_lat'], dff.iloc[row]['location_long']],
                    children=[
                        dl.Tooltip(str(dff.iloc[row]['breed'])),
                        dl.Popup([
                            html.H1('Animal Name'),
                            html.P(str(animal_name))
                        ])
                    ]
                )
            ]
        )
    ]


app.run_server(debug=True)
